In [4]:
import pandas as pd
import os

In [2]:
csv_filename = 'eeg_ablation_results_ec.csv'
df_results = pd.read_csv(csv_filename)

# Dicionários de mapeamento para renomear e categorizar as features
categoria_map = {
    'Somente Potência': 'Espectral',
    'Somente PAC': 'Interação entre Frequências',
    'Somente Assimetria (Potência)': 'Lateralização',
    'Somente Hjorth': 'Complexidade',
    'Somente Entropia': 'Complexidade',
    'Todas as Features Combinadas': 'Combinação (Seleção Múltipla)'
}

feature_map = {
    'Somente Potência': 'Potência (Theta, Alfa, Beta, Gama)',
    'Somente PAC': 'PAC (Phase-Amplitude Coupling)',
    'Somente Assimetria (Potência)': 'Assimetrias Hemisféricas',
    'Somente Hjorth': 'Parâmetros de Hjorth',
    'Somente Entropia': 'Entropia',
    'Todas as Features Combinadas': 'Todas as Features Combinadas (Top 5)'
}

df_results['Categoria de Biomarcador'] = df_results['Feature_Category'].map(categoria_map)
df_results['Método de Extração (Feature)'] = df_results['Feature_Category'].map(feature_map)

df_tabela = df_results[[
    'Categoria de Biomarcador', 
    'Método de Extração (Feature)', 
    'Num_Features', 
    'Accuracy', 
    'AUC_ROC'
]].copy()

df_tabela.rename(columns={
    'Num_Features': 'Quantidade de Features', 
    'Accuracy': 'Acurácia'
}, inplace=True)

df_tabela['Acurácia'] = (df_tabela['Acurácia'] * 100).apply(lambda x: f"{x:.2f}%".replace('.', ','))
df_tabela['AUC_ROC'] = df_tabela['AUC_ROC'].apply(lambda x: f"{x:.4f}".replace('.', ','))

ordem_categorias = [
    'Espectral', 
    'Interação entre Frequências', 
    'Lateralização', 
    'Complexidade', 
    'Combinação (Seleção Múltipla)'
]
df_tabela['Categoria de Biomarcador'] = pd.Categorical(
    df_tabela['Categoria de Biomarcador'], 
    categories=ordem_categorias, 
    ordered=True
)
df_tabela = df_tabela.sort_values(['Categoria de Biomarcador', 'Método de Extração (Feature)']).reset_index(drop=True)

display(df_tabela)

,Categoria de Biomarcador,Método de Extração (Feature),Quantidade de Features,Acurácia,AUC_ROC
0,Espectral,"Potência (Theta, Alfa, Beta, Gama)",136,"31,43%","0,2599"
1,Interação entre Frequências,PAC (Phase-Amplitude Coupling),102,"62,86%","0,7072"
2,Lateralização,Assimetrias Hemisféricas,56,"42,86%","0,0625"
3,Complexidade,Entropia,153,"51,43%","0,4836"
4,Complexidade,Parâmetros de Hjorth,51,"34,29%","0,3947"
5,Combinação (Seleção Múltipla),Todas as Features Combinadas (Top 5),5,"40,00%","0,3224"


In [8]:
pasta_features = 'features_extraidas_ec'

arquivos_features = {
    'Potência': 'features_potencia.csv',
    'PAC': 'features_pac.csv',
    'Assimetria': 'features_asymmetry.csv',
    'Hjorth': 'features_hjorth.csv',
    'Entropia': 'features_entropy.csv'
}

dados_csv = []

for categoria, nome_arquivo in arquivos_features.items():

    caminho_arquivo = os.path.join(pasta_features, nome_arquivo)

    if os.path.exists(caminho_arquivo):
        try:
            df = pd.read_csv(caminho_arquivo, nrows=0)
            colunas = df.columns.tolist()
            
            features = [coluna for coluna in colunas if coluna not in ['patient_id', 'pain_score']]
            
            features_formatadas = ", ".join(features)
            quantidade = len(features)
            
            dados_csv.append({
                'Categoria': f"{categoria}",
                'Quantidade de Features': quantidade,
                'Nomes de Todas as Features Geradas': features_formatadas
            })
            
        except Exception as e:
            print(f"Erro ao ler arquivo '{caminho_arquivo}': {e}")
    else:
        print(f"Arquivo '{caminho_arquivo}' não encontrado.")

df_tabela_features = pd.DataFrame(dados_csv)

nome_arquivo_saida = 'tabela_todas_features.csv'

df_tabela_features.to_csv(nome_arquivo_saida, index=False, encoding='utf-8')

display(df_tabela_features)

,Categoria,Quantidade de Features,Nomes de Todas as Features Geradas
0,Potência,136,"Pow_F3_Delta, Pow_F3_Theta, Pow_F3_Low_Alpha, ..."
1,PAC,102,"PAC_F3_Theta-Low_Gamma, PAC_F3_Theta-High_Gamm..."
2,Assimetria,56,"Asym_Pow_F3-F4_Delta, Asym_Pow_F3-F4_Theta, As..."
3,Hjorth,51,"Hjorth_Activity_F3, Hjorth_Mobility_F3, Hjorth..."
4,Entropia,153,"ApEn_Broadband_F3, ApEn_Delta_F3, ApEn_Theta_F..."
